# Phase 4 — Downstream DPO Bias-Propagation (T4 Colab)

Failed-Phase-3 pivot: BOTH comparisons.
- **Arm A**: Policy-RAW vs Policy-HUMAN — does RM-level length bias per se transfer to policy behavior?
- **Arm B**: Policy-RAW vs Policy-REWEIGHT (best reweight seed, r≈+0.289 @ seed0) — does a ~9% RM-level reduction change downstream behavior?

All three policies start from ONE shared SFT checkpoint. Seeds 42 and 0 per arm. Pre-registered decision table, verbatim. Kill-switch: stop if a DPO run is unstable after one LR halving.

`Runtime → Change runtime type → T4 GPU`, then Run all.


## 1. GPU + pinned deps


In [ ]:
import torch
assert torch.cuda.is_available(), 'Set Runtime -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
%pip install -q 'transformers==5.9.0' 'trl==1.5.0' 'datasets>=2.14.0' 'accelerate>=0.27.0' 'scipy' 'numpy'


## 2. Clone repo + mount Drive


In [ ]:
import os, subprocess
REPO='/content/rlhf-bias-decomp'; BRANCH='phase3-decomposition'
try:
    from google.colab import userdata; token=userdata.get('GITHUB_TOKEN')
except Exception:
    import getpass; token=getpass.getpass('GitHub token: ')
URL=f'https://{token}@github.com/moisheu/rlhf-bias-decomp.git'
if os.path.exists(REPO):
    subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',REPO,'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',REPO,'pull','origin',BRANCH],check=True)
else:
    subprocess.run(['git','clone','--branch',BRANCH,URL,REPO],check=True)
os.chdir(REPO); print(subprocess.run(['git','log','--oneline','-1'],capture_output=True,text=True).stdout)


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive/rlhf-bias-decomp/phase4'
os.makedirs(DRIVE, exist_ok=True); print('Drive:', DRIVE)


## 3. Prepare relabeler RMs (self-contained)

Retrains the two relabeler RMs on this box so both come from one environment, then
evals each to confirm its pooled length-r. (Raw mixed RM ≈ +0.32; reweight seed0 ≈ +0.289.)
Reweight needs the Phase 3 tags+weights, recomputed here (deterministic).


In [ ]:
import os, subprocess, sys
# Phase 3 tags + weights (deterministic; needed by the reweight RM)
subprocess.run([sys.executable,'-m','experiments.decomposition.build_subset_tags'],check=True)
subprocess.run([sys.executable,'-m','experiments.decomposition.compute_weights'],check=True)
# Raw mixed RM (seed 42)
if not os.path.exists('results/reward_model_mixed_seed42/model.safetensors'):
    subprocess.run([sys.executable,'-m','src.train_reward_model'],
                   env={**os.environ,'DATA_MODE':'mixed','TRAIN_SEED':'42'},check=True)
# Reweight RM (seed 0) — the Phase-4 corrected relabeler
if not os.path.exists('results/reward_model_reweight_seed0/model.safetensors'):
    subprocess.run([sys.executable,'-m','experiments.decomposition.train_phase3'],
                   env={**os.environ,'METHOD':'reweight','TRAIN_SEED':'0','TRAIN_BATCH':'16'},check=True)
# confirm their length-r
for d,l in [('results/reward_model_mixed_seed42','raw_relabeler'),
            ('results/reward_model_reweight_seed0','reweight_relabeler')]:
    subprocess.run([sys.executable,'-m','experiments.decomposition.eval_length_correlation',
                    '--model-dir',d,'--label',l,'--out','results/phase4_relabeler_rms.json'],check=True)


## 4. Build Phase 4 data + relabel (Day 1)

Disjoint SFT/DPO 5k slices; relabel the DPO 5k with each RM (+human). Reports agreement & raw-vs-reweight disagreement.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','experiments.dpo.build_phase4_data'],check=True)
subprocess.run([sys.executable,'-m','experiments.dpo.relabel','--labeler','human'],check=True)
subprocess.run([sys.executable,'-m','experiments.dpo.relabel','--labeler','raw',
                '--model-dir','results/reward_model_mixed_seed42'],check=True)
subprocess.run([sys.executable,'-m','experiments.dpo.relabel','--labeler','reweight',
                '--model-dir','results/reward_model_reweight_seed0'],check=True)


## 5. Shared SFT base (Day 1)


In [ ]:
import os, subprocess, sys, shutil
if not os.path.exists('results/phase4/sft_base/model.safetensors'):
    subprocess.run([sys.executable,'-m','experiments.dpo.train_sft'],check=True)
# persist sft_base to Drive so DPO can resume without re-SFT
shutil.copytree('results/phase4/sft_base', f'{DRIVE}/sft_base', dirs_exist_ok=True)
print('sft_base ready + synced to Drive')


## 6. DPO runs (Day 2-3) — 3 policies × 2 seeds, with kill-switch

Kill-switch: if a run is unstable (NaN/inf loss, exit 2), retry ONCE at half LR; a second
instability stops Phase 4 (do not tune). raw_seed42 runs first (the Day-2 canary).


In [ ]:
import os, subprocess, sys
def run_dpo(labeler, seed):
    for i, lr in enumerate([5e-6, 2.5e-6]):
        env={**os.environ,'LABELER':labeler,'TRAIN_SEED':str(seed),'DPO_LR':str(lr),'TRAIN_BATCH':'4'}
        rc=subprocess.run([sys.executable,'-u','-m','experiments.dpo.train_dpo'],env=env).returncode
        if rc==0: return True
        if rc==2:
            print(f'  {labeler}_seed{seed} UNSTABLE at lr={lr}' + (' -> halving LR' if i==0 else ' -> KILL-SWITCH'))
            continue
        raise RuntimeError(f'{labeler}_seed{seed} DPO failed rc={rc}')
    return False
ARMS=[('raw',42),('human',42),('reweight',42),('raw',0),('human',0),('reweight',0)]
for lab,s in ARMS:
    if os.path.exists(f'results/dpo_{lab}_seed{s}/model.safetensors'):
        print(f'SKIP dpo_{lab}_seed{s} (exists)'); continue
    print(f'\n===== DPO {lab}_seed{s} =====', flush=True)
    if not run_dpo(lab,s):
        raise SystemExit(f'KILL-SWITCH: {lab}_seed{s} unstable after one LR halving. Stopping Phase 4 (do not tune).')
print('\nAll DPO runs done.')


## 7. Generate (Day 4) — 6 policies + SFT baseline, cross-scored under both RMs


In [ ]:
import os, subprocess, sys, shutil
RAW_RM='results/reward_model_mixed_seed42'; RW_RM='results/reward_model_reweight_seed0'
def gen(policy_dir, label):
    if os.path.exists(f'results/phase4/gen_{label}.json'):
        print(f'SKIP gen_{label} (exists)'); return
    subprocess.run([sys.executable,'-u','-m','experiments.dpo.generate',
                    '--policy-dir',policy_dir,'--label',label,
                    '--raw-rm',RAW_RM,'--reweight-rm',RW_RM],check=True)
    shutil.copy(f'results/phase4/gen_{label}.json', f'{DRIVE}/gen_{label}.json')
gen('results/phase4/sft_base','sft')
for lab in ['raw','human','reweight']:
    for s in [42,0]:
        gen(f'results/dpo_{lab}_seed{s}', f'{lab}_seed{s}')


## 8. Decision table (Day 5) — both arms


In [ ]:
import subprocess, sys, shutil
subprocess.run([sys.executable,'-m','experiments.dpo.summarize_phase4'])
# persist the small result artifacts
import glob
for f in glob.glob('results/phase4/gen_*.json')+['results/phase4_relabeler_rms.json']:
    try: shutil.copy(f, f'{DRIVE}/'+f.split('/')[-1])
    except Exception as e: print('sync skip', f, e)
